# NEXUS Wake Word Training — Bulletproof Edition

Trains a custom **"nexus"** wake word model using openWakeWord, optimized for:
- **Background noise** (music, traffic, fans, cafe)
- **Low volume / whispered** "nexus" calls
- **Far-field** (speaking from across the room)
- **Soundalike rejection** ("lexus", "texas", "next is" should NOT trigger)

## Kaggle Setup

1. **Settings -> Accelerator -> GPU T4 x2** (or P100)
2. **Settings -> Internet -> On**
3. **Run all cells** (~90 minutes total)
4. Download `nexus.onnx` from output

## Issues Fixed (from Colab failures)

| Issue | Fix |
|-------|-----|
| `torch==1.13.1` no Python 3.12 wheels | Use pre-installed `torch>=2.0` |
| `torchaudio` deprecated APIs | Monkey-patch with soundfile |
| `setuptools 82+` removed `pkg_resources` | Pin `setuptools<82` |
| `pyarrow`/`fsspec` break `datasets` | Pin versions |
| `piper_sample_generator` API mismatch | Use `piper-tts` PiperVoice directly |
| AudioSet 404 | Use FMA + synthetic noise |
| Sample rate mismatch (Piper 22050) | Resample in patched torchaudio.load |
| `onnxscript` missing | Install explicitly |
| `webrtcvad` C compilation | Install build-essential first |

## 1. Install System Dependencies

Build tools and audio libraries for C extensions (webrtcvad, soundfile).

In [ ]:
import subprocess, sys, os

# ─── Install system packages ───
try:
    subprocess.check_call(["apt-get", "update", "-qq"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([
        "apt-get", "install", "-y", "-qq",
        "build-essential", "cmake", "espeak-ng", "libespeak-ng-dev",
        "libsndfile1", "pkg-config", "ffmpeg", "unzip",
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("System packages installed")
except Exception as e:
    print(f"apt-get note: {e}")

# ─── Check GPU capability WITHOUT importing torch ───
# We use nvidia-smi to detect the GPU, then install the right PyTorch.
# This avoids the importlib.reload(torch) problem (PyTorch C++ extensions
# cannot be reloaded in the same process).
gpu_name = ""
gpu_compatible = False
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True, timeout=10)
    gpu_name = result.stdout.strip()
    print(f"GPU detected via nvidia-smi: {gpu_name}")
except Exception:
    print("No nvidia-smi — assuming CPU-only")

# P100 = sm_60 (needs PyTorch cu118), T4 = sm_75, V100 = sm_70, A100 = sm_80
# Pre-installed PyTorch 2.10+cu128 only supports sm_70+
needs_pytorch_fix = "P100" in gpu_name

if needs_pytorch_fix:
    print(f"\nP100 detected (sm_60) — pre-installed PyTorch 2.10+cu128 won't work.")
    print("Installing PyTorch 2.4.1 + CUDA 11.8 (supports sm_37 to sm_90, includes P100)...")
    # Uninstall the incompatible torch AND all torch-* extensions that link
    # against the old torch C++ ABI (torchcodec, torchvision, torchtriton, etc.)
    # If we leave them installed, they'll fail with "undefined symbol" errors
    # when torchaudio tries to load them.
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
        "torch", "torchaudio", "torchvision", "torchtriton",
        "torchcodec", "torchdata", "torchtext",
        "spacy", "spacy-legacy", "spacy-loggers", "spacy-alignments",
        "spacy-ml", "spacy-transformers", "thinc"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.4.1", "torchaudio==2.4.1", "torchvision==0.19.1",
        "--index-url", "https://download.pytorch.org/whl/cu118"])
    print("PyTorch 2.4.1+cu118 installed successfully")
else:
    print("GPU is compatible with pre-installed PyTorch (or no GPU)")

# NOW import torch (after any reinstall)
import torch
print(f"\nPython: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"CUDA capability: sm_{cap[0]}{cap[1]}")
    print(f"VRAM: {vram:.1f} GB")
    # Test actual GPU computation
    try:
        x = torch.randn(100, 100, device="cuda")
        y = x @ x
        print(f"GPU computation test: PASSED (result shape: {y.shape})")
        gpu_compatible = True
    except Exception as e:
        print(f"GPU computation test: FAILED — {e}")
        print("Falling back to CPU training")
        os.environ["CUDA_VISIBLE_DEVICES"] = ""
else:
    print("No GPU — using CPU")

print(f"\nGPU compatible for training: {gpu_compatible}")

## 2. Install Python Dependencies

All versions pinned for Kaggle Python 3.12 + PyTorch 2.x.

In [ ]:
import subprocess, sys, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])

# Compat fixes FIRST
pip_install("setuptools<82")
pip_install("pyarrow<15.0.0")
pip_install("fsspec<2024.1.0")

# Core
pip_install("soundfile", "scipy", "mutagen==1.47.0")

# Pin datasets<4.0 — newer versions require torchcodec
pip_install("datasets>=2.14,<4.0")

# openWakeWord from git — this works (v13 proved it).
# The PyPI version requires tflite-runtime (unavailable on Python 3.12 Linux),
# but the git repo's setup.py doesn't have that requirement.
if not os.path.exists("openwakeword_src"):
    subprocess.check_call(["git", "clone", "--depth", "1",
        "https://github.com/dscripka/openWakeWord.git", "openwakeword_src"])
pip_install("-e", "./openwakeword_src")

# piper_sample_generator deps
pip_install("piper-phonemize", "-f", "https://k2-fsa.github.io/icefall/piper_phonemize.html")
pip_install("piper-tts==1.3.0")

# Training deps
pip_install("torchinfo==1.8.0", "torchmetrics==1.2.0")
pip_install("speechbrain>=0.5.14")
pip_install("audiomentations==0.33.0", "torch-audiomentations==0.11.0")
pip_install("acoustics==0.2.6")

# pronouncing — required by openwakeword.data
pip_install("pronouncing")

# ONNX export — pin onnxscript to 0.1.0 (has ParamSchema)
pip_install("onnx==1.16.1", "onnxscript==0.1.0")
pip_install("onnxruntime>=1.16")

# Data utils
pip_install("webrtcvad", "pyyaml", "tqdm", "requests")

print("All Python dependencies installed")

## 3. Apply Compatibility Patches

Must be applied BEFORE importing openwakeword or speechbrain.

In [ ]:
import logging, subprocess, sys, tempfile, os, requests
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="  %(levelname)-8s %(message)s")

def patch_pkg_resources():
    try:
        import pkg_resources; return "ok"
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "setuptools<82", "-q"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return "applied"

def patch_torchaudio_load():
    import torch, torchaudio
    if getattr(torchaudio, "_oww_patched", False): return "ok"
    def _load(filepath, *a, **kw):
        import numpy as np, soundfile as sf
        data, sr = sf.read(str(filepath), dtype="float32")
        if data.ndim == 1: data = data[np.newaxis, :]
        else: data = data.T
        if sr != 16000:
            from scipy.signal import resample
            new_len = int(data.shape[-1] * 16000 / sr)
            if data.ndim == 2:
                data = np.stack([resample(data[c], new_len).astype(np.float32) for c in range(data.shape[0])])
            else:
                data = resample(data, new_len).astype(np.float32)
            sr = 16000
        return torch.from_numpy(data), sr
    torchaudio.load = _load
    torchaudio._oww_patched = True
    return "applied"

def patch_torchaudio_info():
    import torchaudio
    if getattr(torchaudio, "_oww_info_patched", False): return "ok"
    class AMD:
        __slots__ = ("sample_rate", "num_frames", "num_channels", "bits_per_sample", "encoding")
        def __init__(self, sr, nf, nc):
            self.sample_rate = sr; self.num_frames = nf; self.num_channels = nc
            self.bits_per_sample = 16; self.encoding = "PCM_S"
    def _info(fp):
        import soundfile as sf
        fi = sf.info(str(fp))
        return AMD(fi.samplerate, fi.frames, fi.channels)
    torchaudio.info = _info
    if not hasattr(torchaudio, "AudioMetaData"): torchaudio.AudioMetaData = AMD
    torchaudio._oww_info_patched = True
    return "applied"

def patch_torchaudio_backends():
    import torchaudio
    if hasattr(torchaudio, "list_audio_backends"): return "ok"
    torchaudio.list_audio_backends = lambda: ["soundfile"]
    return "applied"

for name, fn in [("pkg_resources", patch_pkg_resources),
                 ("torchaudio.load", patch_torchaudio_load),
                 ("torchaudio.info", patch_torchaudio_info),
                 ("torchaudio.backends", patch_torchaudio_backends)]:
    try: print(f"  {name}: {fn()}")
    except Exception as e: print(f"  {name}: FAILED: {e}")

# ─── Download openWakeWord ONNX model resources manually ───
# The AudioFeatures class in utils.py expects .onnx versions of:
#   melspectrogram.onnx  — for computing mel spectrograms
#   embedding_model.onnx — for computing audio embeddings
# These are not in the git repo and download_models() doesn't exist in the
# git version. We download them from the GitHub releases.
models_dir = Path("openwakeword_src/openwakeword/resources/models")
models_dir.mkdir(parents=True, exist_ok=True)

onnx_models = {
    "melspectrogram.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",
    "embedding_model.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
}

for name, url in onnx_models.items():
    dest = models_dir / name
    if dest.exists() and dest.stat().st_size > 1000:
        print(f"  {name}: exists ({dest.stat().st_size/1024:.0f} KB)")
    else:
        print(f"  Downloading {name} from {url}...")
        resp = requests.get(url, stream=True, timeout=120)
        resp.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1<<20):
                f.write(chunk)
        print(f"  {name}: downloaded ({dest.stat().st_size/1024:.0f} KB)")

# Verify
for name in onnx_models:
    p = models_dir / name
    assert p.exists() and p.stat().st_size > 1000, f"{name} missing!"
print("openWakeWord ONNX models ready")

# Verify torchaudio patches
import torch, torchaudio, numpy as np, soundfile as sf
with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
    sf.write(f.name, np.zeros(22050, dtype=np.float32), 22050)
    wav, sr = torchaudio.load(f.name)
    assert sr == 16000, f"Expected 16000, got {sr}"
    os.unlink(f.name)
print("All patches verified")

## 4. Download Piper TTS Voice Model

We use the Piper ONNX voice model directly with `PiperVoice` — this is more
reliable than the `piper_sample_generator` which requires `piper_train`.

The `en_US-lessac-medium.onnx` voice is a clear, neutral American English voice.

In [ ]:
import os, requests, subprocess
from pathlib import Path

# ─── Clone piper_sample_generator at the commit that has generate_samples.py ───
# The latest version moved generate_samples into __main__.py, but openWakeWord's
# train.py does `from generate_samples import generate_samples`, which needs the
# old-style generate_samples.py file. Commit 9c1019c is the last commit with it.
psg_path = "piper_sample_generator_src"
if not os.path.exists(psg_path):
    subprocess.check_call(["git", "clone",
        "https://github.com/rhasspy/piper-sample-generator.git", psg_path])
    # Checkout the commit that has generate_samples.py
    subprocess.check_call(["git", "checkout", "9c1019c"], cwd=psg_path)
print(f"piper_sample_generator cloned at commit 9c1019c: {psg_path}")

# Verify generate_samples.py exists
gs_file = Path(psg_path) / "generate_samples.py"
assert gs_file.exists(), f"generate_samples.py not found in {psg_path}!"
print(f"generate_samples.py found: {gs_file}")

# ─── Download piper_sample_generator model (.pt file) ───
psg_dir = Path(psg_path) / "models"
psg_dir.mkdir(exist_ok=True)
pt_model = psg_dir / "en_US-libritts_r-medium.pt"

if not pt_model.exists() or pt_model.stat().st_size < 1_000_000:
    url = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
    print(f"Downloading piper_sample_generator model from {url}...")
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    with open(pt_model, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1<<20):
            f.write(chunk)
    print(f"  {pt_model.stat().st_size/1e6:.1f} MB")
else:
    print(f"piper_sample_generator model exists: {pt_model.stat().st_size/1e6:.1f} MB")

# ─── Also download Piper ONNX voice (for any custom generation) ───
voice_dir = Path("piper_voices")
voice_dir.mkdir(exist_ok=True)

voice_name = "en_US-lessac-medium"
base_url = f"https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/"

onnx_path = voice_dir / f"{voice_name}.onnx"
json_path = voice_dir / f"{voice_name}.onnx.json"

for path, suffix in [(onnx_path, f"{voice_name}.onnx"), (json_path, f"{voice_name}.onnx.json")]:
    if not path.exists():
        url = base_url + suffix
        print(f"Downloading {path.name}...")
        resp = requests.get(url, stream=True, timeout=120)
        resp.raise_for_status()
        with open(path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1<<20):
                f.write(chunk)
        print(f"  {path.stat().st_size/1e6:.1f} MB")
    else:
        print(f"Exists: {path.name}")

# Save the path for the config cell
with open("psg_path.txt", "w") as f:
    f.write(psg_path)
print(f"\npsg_path saved: {psg_path}")

## 5. Download Augmentation Datasets

- MIT Room Impulse Responses (far-field simulation)
- Background noise (MUSAN/synthetic)
- ACAV100M pre-computed negative features (11 hours of speech)
- Validation features for early stopping

In [ ]:
import os, numpy as np, soundfile as sf, requests
from pathlib import Path
from tqdm import tqdm
import subprocess

data_dir = Path("training_data")
data_dir.mkdir(exist_ok=True)
SR = 16000

# ─── MIT RIRs ───
rir_dir = data_dir / "mit_rirs"
if not rir_dir.exists() or len(list(rir_dir.glob("*.wav"))) < 100:
    rir_dir.mkdir(parents=True, exist_ok=True)
    print("Downloading MIT RIRs...")
    from datasets import load_dataset
    rir_ds = load_dataset("davidscripka/MIT_environmental_impulse_responses",
                          split="train", streaming=True)
    import scipy.io
    for row in tqdm(rir_ds, desc="RIRs"):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(str(rir_dir / name), 16000,
                               (row['audio']['array']*32767).astype(np.int16))
print(f"RIRs: {len(list(rir_dir.glob('*.wav')))}")

# ─── Noise/background audio ───
# Use the FSD50K sample from openWakeWord resources (smaller, reliable)
noise_dir = data_dir / "noise"
if not noise_dir.exists() or len(list(noise_dir.glob("*.wav"))) < 50:
    noise_dir.mkdir(parents=True, exist_ok=True)
    print("Downloading noise sample from openWakeWord resources...")
    url = "https://f002.backblazeb2.com/file/openwakeword-resources/data/fsd50k_sample.zip"
    zip_path = data_dir / "fsd50k_sample.zip"
    resp = requests.get(url, stream=True, timeout=300)
    if resp.status_code == 200:
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1<<20):
                f.write(chunk)
        print(f"  Downloaded {zip_path.stat().st_size/1e6:.1f} MB, extracting...")
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(data_dir)
        zip_path.unlink()
        # Move wavs into noise_dir
        fsd_dir = data_dir / "fsd50k_sample"
        if fsd_dir.exists():
            for wav in fsd_dir.glob("*.wav"):
                wav.rename(noise_dir / wav.name)
            fsd_dir.rmdir() if fsd_dir.exists() and not any(fsd_dir.iterdir()) else None
    else:
        print(f"  FSD50K download failed (HTTP {resp.status_code}), generating synthetic noise...")
        # Fallback: generate synthetic white/pink noise
        rng = np.random.default_rng(42)
        for i in tqdm(range(200), desc="Synthetic noise"):
            dur = rng.uniform(2, 10)
            n = int(SR * dur)
            # Mix of white noise, pink-ish noise, and tones
            white = rng.standard_normal(n) * 0.1
            # Simple pink noise filter
            pink = np.cumsum(white)
            pink = pink / (np.abs(pink).max() + 1e-9) * 0.05
            tone_freq = rng.uniform(100, 2000)
            tone = np.sin(2*np.pi*tone_freq*np.arange(n)/SR) * 0.02
            mix = white + pink + tone
            sf.write(str(noise_dir / f"synth_noise_{i:04d}.wav"), mix.astype(np.float32), SR)

print(f"Noise clips: {len(list(noise_dir.glob('*.wav')))}")

# ─── Precomputed openWakeWord features (REQUIRED by train.py) ───
# Training features: ~2000 hours from ACAV100M Dataset
acav_path = data_dir / "acav100m_features.npy"
if not acav_path.exists() or acav_path.stat().st_size < 1_000_000:
    print("Downloading ACAV100M precomputed features (~2.5GB)...")
    url = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    resp = requests.get(url, stream=True, timeout=600)
    resp.raise_for_status()
    with open(acav_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1<<20):
            f.write(chunk)
    print(f"  {acav_path.stat().st_size/1e9:.2f} GB")
else:
    print(f"ACAV100M features exist: {acav_path.stat().st_size/1e9:.2f} GB")

# Validation features for false positive rate estimation (~11 hours)
val_path = data_dir / "validation_features.npy"
if not val_path.exists() or val_path.stat().st_size < 1_000_000:
    print("Downloading validation features...")
    url = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy"
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    with open(val_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1<<20):
            f.write(chunk)
    print(f"  {val_path.stat().st_size/1e6:.1f} MB")
else:
    print(f"Validation features exist: {val_path.stat().st_size/1e6:.1f} MB")

print("All data ready")

## 6-9. Data Generation (handled by train.py)

**SKIPPED** — The custom data generation cells (Piper TTS, augmentation, train/val split) 
are no longer needed. The openWakeWord `train.py` script handles the entire pipeline 
when called with `--generate_clips --augment_clips --train_model`:

1. **`--generate_clips`**: Uses `piper_sample_generator` to synthesize positive/negative clips
2. **`--augment_clips`**: Applies room impulse responses + background noise, computes mel features
3. **`--train_model`**: Trains the DNN model on the computed features

The custom cells (12-18) are kept below for reference but are not executed.

In [ ]:
pass  # Skipped — train.py handles data generation


## 7. Generate Adversarial Negative Clips (2,000+)

Phrases that should NOT trigger the wake word.

In [ ]:
pass  # Skipped — train.py handles data generation


## 8. Augment Positive Clips (4x)

Apply: volume variation, speed variation, room reverberation (RIRs), background noise mixing.
Creates 4,000+ augmented clips from 1,000 base clips.

In [ ]:
pass  # Skipped — train.py handles data generation


## 9. Split into Train/Validation (90/10)

In [ ]:
pass  # Skipped — train.py handles data generation


## 10. Create Training Configuration

In [ ]:
import yaml
from pathlib import Path

# Read the piper_sample_generator path saved by cell 8
psg_path = Path("psg_path.txt").read_text().strip() if Path("psg_path.txt").exists() else "piper_sample_generator_src"

config = {
    # ─── Data generation (used by --generate_clips) ───
    "piper_sample_generator_path": psg_path,
    "target_phrase": ["nexus", "hey nexus", "nexus wake up"],
    "custom_negative_phrases": [
        "lexus", "texas", "next is", "neck us",
        "hey siri", "ok google", "alexa", "hey cortana",
    ],
    "n_samples": 5000,           # positive training clips
    "n_samples_val": 500,        # positive validation clips
    "tts_batch_size": 32,        # batch size for TTS generation

    # ─── Augmentation (used by --augment_clips) ───
    "total_length": 20480,       # 1280 samples * 16 chunks = ~1.28s at 16kHz
    "augmentation_batch_size": 16,
    "augmentation_rounds": 1,
    "rir_paths": ["./training_data/mit_rirs"],
    "background_paths": ["./training_data/noise"],
    "background_paths_duplication_rate": [1],

    # ─── Model training (used by --train_model) ───
    "model_type": "dnn",
    "layer_size": 32,
    "steps": 50000,
    "batch_n_per_class": {
        "positive": 50,
        "adversarial_negative": 50,
        "ACAV100M_sample": 1024,
    },
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,
    "early_stopping": True,

    # ─── Precomputed features (REQUIRED by train.py) ───
    "feature_data_files": {
        "ACAV100M_sample": "./training_data/acav100m_features.npy",
    },
    "false_positive_validation_data_path": "./training_data/validation_features.npy",
    "validation_features": "./training_data/validation_features.npy",

    # ─── Output ───
    "output_dir": "./nexus_model",
    "model_name": "nexus",
}

with open("nexus_config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print("Config written")
print(yaml.dump(config, default_flow_style=False))

## 11. Train the Wake Word Model

Runs openWakeWord's `train.py` with our config. Takes ~30-60 min on T4 GPU.

In [ ]:
import subprocess, sys, os
from pathlib import Path

Path("nexus_model").mkdir(exist_ok=True)

# Find train.py
train_script = None
for p in Path("openwakeword_src").rglob("train.py"):
    train_script = p; break

if not train_script:
    raise FileNotFoundError("train.py not found in openwakeword_src/")

print(f"Training script: {train_script}")
print("Starting full pipeline: generate_clips -> augment_clips -> train_model")
print("=" * 60)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# openWakeWord train.py arguments:
#   --training_config (required) — path to YAML config
#   --generate_clips  — generate synthetic speech with piper_sample_generator
#   --augment_clips   — augment clips with RIRs + background noise, compute features
#   --train_model     — train the DNN model
#   --overwrite       — overwrite existing features
#
# NOTE: train.py also tries to convert ONNX -> TFLite at the end, which fails
# because onnx_tf is not installed. That's fine — we only need the ONNX model.
# The ONNX file is saved BEFORE the TFLite conversion attempt.
result = subprocess.run(
    [sys.executable, str(train_script),
     "--training_config", "nexus_config.yaml",
     "--generate_clips",
     "--augment_clips",
     "--train_model",
     "--overwrite"],
    env=env, timeout=3600,
)
print(f"Training exited with code: {result.returncode}")

# Check if ONNX was produced despite nonzero exit (TFLite conversion failure)
onnx_path = Path("nexus_model/nexus.onnx")
if onnx_path.exists() and onnx_path.stat().st_size > 1000:
    print(f"ONNX model exists: {onnx_path} ({onnx_path.stat().st_size/1024:.0f} KB)")
    print("Training SUCCEEDED (exit code was from TFLite conversion, which we don't need)")
elif result.returncode != 0:
    raise RuntimeError(f"openWakeWord training failed with exit code {result.returncode}")

print("=" * 60)
print("Training complete")

## 12. Export Model to ONNX

In [ ]:
import os, sys
from pathlib import Path

model_dir = Path("nexus_model")

# Find ONNX file (training may auto-export)
onnx_files = list(model_dir.rglob("*.onnx"))
# Also check for .pt checkpoints
pt_files = sorted(model_dir.rglob("*.pt")) + sorted(model_dir.rglob("*.pth"))

print(f"ONNX files: {len(onnx_files)}")
for f in onnx_files: print(f"  {f} ({f.stat().st_size/1024:.0f} KB)")
print(f"PT files: {len(pt_files)}")
for f in pt_files: print(f"  {f} ({f.stat().st_size/1024:.0f} KB)")

if onnx_files:
    onnx_path = onnx_files[0]
    print(f"Using existing ONNX: {onnx_path}")
elif pt_files:
    # Export from PyTorch checkpoint
    import torch
    ckpt = pt_files[-1]
    print(f"Exporting from checkpoint: {ckpt}")

    sys.path.insert(0, str(Path("openwakeword_src").resolve()))
    try:
        from openwakeword.model import Model
        model = Model(wakeword_models=[str(ckpt)])
        onnx_path = model_dir / "nexus.onnx"

        # Get underlying model
        if hasattr(model, 'models') and model.models:
            pt_model = list(model.models.values())[0]
        elif hasattr(model, 'model'):
            pt_model = model.model
        else:
            pt_model = model

        dummy = torch.randn(1, 16, 96)
        torch.onnx.export(
            pt_model if hasattr(pt_model, 'forward') else pt_model,
            dummy, str(onnx_path),
            opset_version=14,
            input_names=['input'], output_names=['output'],
            dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        )
        print(f"Exported: {onnx_path} ({onnx_path.stat().st_size/1024:.0f} KB)")
    except Exception as e:
        print(f"Export failed: {e}")
        # Try openwakeword utils
        try:
            from openwakeword.utils import export_onnx
            onnx_path = model_dir / "nexus.onnx"
            export_onnx(str(ckpt), str(onnx_path))
            print(f"Exported via utils: {onnx_path}")
        except Exception as e2:
            print(f"Utils export also failed: {e2}")
            raise
else:
    # List all files for debugging
    print("No model files found. All files in nexus_model/:")
    for f in model_dir.rglob("*"):
        if f.is_file(): print(f"  {f} ({f.stat().st_size/1024:.0f} KB)")
    raise FileNotFoundError("No model checkpoint or ONNX file found")

# Verify ONNX
import onnx
onnx.checker.check_model(onnx.load(str(onnx_path)))
print(f"ONNX verified: {onnx_path} ({onnx_path.stat().st_size/1024:.0f} KB)")

## 13. Test the Trained Model

In [ ]:
import numpy as np, soundfile as sf
from pathlib import Path
from scipy.signal import resample
import subprocess, sys

onnx_path = Path("nexus_model/nexus.onnx")
assert onnx_path.exists(), f"ONNX model not found: {onnx_path}"
print(f"Model: {onnx_path} ({onnx_path.stat().st_size/1024:.0f} KB)")

# Use openWakeWord's AudioFeatures in a subprocess to compute features correctly
# (can't import directly in notebook due to sys.path caching)
feature_script = """
import sys, json, numpy as np, soundfile as sf
from scipy.signal import resample

sys.path.insert(0, "openwakeword_src")
from openwakeword.utils import AudioFeatures
import onnxruntime as ort

# Load models
F = AudioFeatures()
sess = ort.InferenceSession("nexus_model/nexus.onnx")
input_name = sess.get_inputs()[0].name

def score_clip(audio_16k):
    # AudioFeatures expects int16-scale float32
    audio_scaled = (audio_16k * 32768.0).astype(np.float32)
    # Get embeddings (shape: n_frames x 96)
    embeddings = F._get_embeddings(audio_scaled)
    # Model expects (batch, 16, 96) — take last 16 frames
    if embeddings.shape[0] < 16:
        # Pad if too short
        embeddings = np.pad(embeddings, ((16 - embeddings.shape[0], 0), (0, 0)))
    features = embeddings[-16:].reshape(1, 16, 96).astype(np.float32)
    return float(sess.run(None, {input_name: features})[0][0])

results = {"positive": [], "negative": [], "silence": None}

# Positive tests
from pathlib import Path
pos_files = sorted(Path("training_data/val/positive").glob("*.wav"))[:10]
if not pos_files:
    pos_files = sorted(Path("nexus_model/nexus/positive_test").glob("*.wav"))[:10]
for f in pos_files:
    audio, sr = sf.read(str(f), dtype="float32")
    if sr != 16000: audio = resample(audio, int(len(audio) * 16000 / sr)).astype(np.float32)
    if audio.ndim > 1: audio = audio[:, 0]
    s = score_clip(audio)
    results["positive"].append({"file": f.name, "score": s})
    print(f"  POS {f.name}: {s:.3f}")

# Negative tests
neg_files = sorted(Path("training_data/val/negative").glob("*.wav"))[:10]
if not neg_files:
    neg_files = sorted(Path("nexus_model/nexus/negative_test").glob("*.wav"))[:10]
for f in neg_files:
    audio, sr = sf.read(str(f), dtype="float32")
    if sr != 16000: audio = resample(audio, int(len(audio) * 16000 / sr)).astype(np.float32)
    if audio.ndim > 1: audio = audio[:, 0]
    s = score_clip(audio)
    results["negative"].append({"file": f.name, "score": s})
    print(f"  NEG {f.name}: {s:.3f}")

# Silence
silence_score = score_clip(np.zeros(16000, dtype=np.float32))
results["silence"] = silence_score
print(f"  SILENCE: {silence_score:.3f}")

# Summary
pos_scores = [r["score"] for r in results["positive"]]
neg_scores = [r["score"] for r in results["negative"]]
print(f"\\n=== Summary ===")
if pos_scores:
    print(f"  Positive avg: {np.mean(pos_scores):.3f} (target > 0.5)")
if neg_scores:
    print(f"  Negative avg: {np.mean(neg_scores):.3f} (target < 0.3)")
print(f"  Silence:      {silence_score:.3f} (target < 0.1)")

# Save results
with open("test_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved to test_results.json")
"""

print("Running model evaluation in subprocess...")
result = subprocess.run(
    [sys.executable, "-c", feature_script],
    capture_output=True, text=True, timeout=300
)
print(result.stdout)
if result.returncode != 0:
    print(f"STDERR: {result.stderr[:1000]}")
    print("Evaluation failed, but model was trained. Check nexus_model/nexus.onnx")

## 14. Download and Deploy

1. Download `nexus_model/nexus.onnx` from Kaggle output
2. Replace `src-tauri/resources/oww/nexus.onnx` in your NEXUS project
3. Rebuild: `pwsh ./scripts/build.ps1`
4. Test: say "nexus" at various volumes and with background noise

In [ ]:
from pathlib import Path

onnx_path = Path("nexus_model/nexus.onnx")
if onnx_path.exists():
    print(f"Model ready: {onnx_path}")
    print(f"  Size: {onnx_path.stat().st_size/1024:.0f} KB")
    print()
    print("Download from Kaggle output panel.")
    print("Replace: src-tauri/resources/oww/nexus.onnx")
    print("Rebuild: pwsh ./scripts/build.ps1")
else:
    onnx_files = list(Path(".").rglob("*.onnx"))
    custom = [f for f in onnx_files if "melspectrogram" not in f.name
              and "embedding" not in f.name and "vad" not in f.name.lower()]
    if custom:
        print(f"Found: {custom[0]} ({custom[0].stat().st_size/1024:.0f} KB)")
    else:
        print("No ONNX model found. Check training output above.")